# AgentOps Lab 04 - Tool engineering

This notebook turns a vague, dangerous operations tool into narrow, validated, auditable tools. It demonstrates why agent reliability often depends less on clever prompting and more on tool boundaries.



## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — Tool engineering and safe contracts

### Concepts to master

- narrow tools versus broad admin APIs
- typed validation and permission boundaries
- retryable versus terminal failures

### Implementation walkthrough

Use `tool_engineering.py` to compare `admin_api(command)` with narrow tools for logs, deployments, restarts, and incident tickets. The useful design work happens before the model sees a tool.

### Deliberate failure case

Let a broad command string tool accept `restart checkout now`. Notice how validation, authorization, audit, and retry semantics become ambiguous.

### Learner exercise

Add a `rollback_deployment(service, deployment_id, reason)` request model and require incident ID, minimum reason length, and human approval.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## The dangerous tool

```python
admin_api(command: str)
```

This single tool can query logs, restart services, delete records, deploy software, send notifications, and change config. It is dangerous because the schema hides intent. A model can put anything in the command string, authorization cannot easily distinguish read-only from destructive actions, and failures are hard to classify.

```mermaid
flowchart TD
    A["admin_api(command)"] --> B["query logs"]
    A --> C["restart service"]
    A --> D["delete records"]
    A --> E["deploy software"]
    A --> F["send notifications"]
    A --> G["change config"]
```


In [ ]:
from pathlib import Path
import sys

repo_root = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / "curriculum" / "advanced" / "05-incident-response-capstone" / "agentops_lab").exists()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the repository checkout.")
sys.path.insert(0, str(repo_root / "curriculum" / "advanced" / "05-incident-response-capstone"))

from agentops_lab.tool_engineering import (
    PermissionDenied,
    RestartRequest,
    admin_api,
    compare_bad_and_good_tools,
    create_incident_ticket,
    query_logs,
    restart_service,
    retry_policy,
    run_with_retry,
)


In [ ]:
admin_api("restart checkout and delete failed payment records")


## Refactor into narrow tools

A better design separates read-only tools from write tools and gives each tool a typed contract:

- `query_logs(service, time_range_minutes, severity)`
- `get_recent_deployments(service)`
- `restart_service(service, reason, incident_id)`
- `create_incident_ticket(title, severity, evidence)`

```mermaid
flowchart LR
    A["Agent request"] --> B{"Read-only?"}
    B -- "Yes" --> C["query_logs / get_recent_deployments"]
    B -- "No" --> D["Validate structured request"]
    D --> E{"Human approval?"}
    E -- "Approved" --> F["restart_service"]
    E -- "Denied or missing" --> G["escalate"]
```


## Structured validation

In a real SDK integration, you would commonly use Pydantic-style validation:

```python
from typing import Literal
from pydantic import BaseModel, Field

class RestartRequest(BaseModel):
    service: Literal["checkout", "payments", "catalog"]
    reason: str = Field(min_length=20)
    incident_id: str
```

This repository keeps the executable lab dependency-free, so the Python module uses an equivalent dataclass validator. The principle is the same: invalid services, short reasons, and malformed incident IDs should fail before a tool touches infrastructure.


In [ ]:
query_logs("checkout", time_range_minutes=60, severity="ERROR")


In [ ]:
create_incident_ticket(
    title="European checkout 3DS failures",
    severity="sev2",
    evidence=["eu-west 3DS callback errors", "active checkout payment incident"],
)


## Simulate tool failures

A reliable tool contract includes predictable failure classes. In this lab:

- `ToolTimeout` and `RateLimit` are retryable.
- `PermissionDenied` escalates to a human.
- `InvalidService` and validation errors stop the run.


In [ ]:
run_with_retry(max_attempts=3, attempts_before_success=2)


In [ ]:
try:
    restart_service(RestartRequest("checkout", "Need approval for regional checkout recovery", "INC-1042"))
except Exception as exc:
    print(type(exc).__name__, "->", retry_policy(exc))


## Exercise

- Add an `update_feature_flag(service, flag_name, desired_state, incident_id)` tool. What validation and approval does it need?
- Add a `RateLimit` simulation and use exponential backoff in the retry policy.
- Write three bad `admin_api` commands and refactor each into a safe narrow tool.
- Decide which tools should be available to a support assistant versus an on-call engineering assistant.

References: [OpenAI Agents SDK tools](https://openai.github.io/openai-agents-python/tools/), [OpenAI Agents SDK guardrails](https://openai.github.io/openai-agents-python/guardrails/), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).
